In [2]:
import numpy as np
import cv2
import math
import os
from pathlib import Path

# ==========================================
# 1. CẤU HÌNH ĐẦU VÀO VÀ THÔNG SỐ
# ==========================================
# Danh sách các file .npz bạn muốn nối liên tục thành một câu/chuỗi ký hiệu
NPZ_FILE_LIST = [
    "C:/Users/ADMIN/qh/processed/processed/npz_output/Tôi_740255.npz",
    "C:/Users/ADMIN/qh/processed/processed/npz_output/Yêu thương_007340.npz",  # Thêm các file tiếp theo vào đây
    "C:/Users/ADMIN/qh/processed/processed/npz_output/Yêu thương_008274.npz",
    "C:/Users/ADMIN/qh/processed/processed/npz_output/Yên tĩnh_013091.npz",
        # "C:/Users/ADMIN/qh/processed/processed/npz_output/Yên tĩnh_011052.npz",  # Thêm các file tiếp theo vào đây
    #     "C:/Users/ADMIN/qh/processed/processed/npz_output/Thứ tư_017009.npz"
]

WIDTH = 800
HEIGHT = 800
PADDING = 100
TRANSITION_SECONDS = 0.25  # Thời gian chuyển tiếp giữa 2 file (giây)

# ==========================================
# 2. KHAI BÁO KẾT NỐI XƯƠNG (CHỈ THÂN TRÊN + TAY)
# ==========================================
body_connections = [
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16), # Vai, tay
    (11, 23), (12, 24), (23, 24),                     # Thân / Hông
]

def get_hand_connections(start_idx):
    return [
        (start_idx, start_idx+1), (start_idx+1, start_idx+2), (start_idx+2, start_idx+3), (start_idx+3, start_idx+4),
        (start_idx, start_idx+5), (start_idx+5, start_idx+6), (start_idx+6, start_idx+7), (start_idx+7, start_idx+8),
        (start_idx, start_idx+9), (start_idx+9, start_idx+10), (start_idx+10, start_idx+11), (start_idx+11, start_idx+12),
        (start_idx, start_idx+13), (start_idx+13, start_idx+14), (start_idx+14, start_idx+15), (start_idx+15, start_idx+16),
        (start_idx, start_idx+17), (start_idx+17, start_idx+18), (start_idx+18, start_idx+19), (start_idx+19, start_idx+20),
        (start_idx+5, start_idx+9), (start_idx+9, start_idx+13), (start_idx+13, start_idx+17)
    ]

# Thiết lập màu sắc xương
connections = []
for u, v in body_connections:
    connections.append((u, v, (255, 255, 0)))    # Thân: Xanh Cyan
for u, v in get_hand_connections(33):
    connections.append((u, v, (0, 0, 255)))      # Tay trái: Đỏ
for u, v in get_hand_connections(54):
    connections.append((u, v, (0, 255, 0)))      # Tay phải: Xanh lá

# ==========================================
# 3. HÀM HỖ TRỢ NỘI SUY CHUYỂN TIẾP (EASE)
# ==========================================
def ease(w):
    return 0.5 - 0.5 * np.cos(np.pi * w)

# ==========================================
# 4. ĐỌC, HIỆU CHỈNH VÀ NỐI CÁC FILE .NPZ
# ==========================================
all_processed_clips = []
global_fps = 30.0  # Mặc định

for path in NPZ_FILE_LIST:
    if not os.path.exists(path):
        print(f"[CẢNH BÁO] Không tìm thấy file: {path}, bỏ qua.")
        continue
    
    data = np.load(path)
    pose = data['pose']     # (Frames, 33, 3)
    lhand = data['lhand']   # (Frames, 21, 3)
    rhand = data['rhand']   # (Frames, 21, 3)
    global_fps = float(data['fps'])
    
    num_f = pose.shape[0]
    clip_frames = []
    
    for i in range(num_f):
        p = pose[i].copy()
        lh = lhand[i].copy()
        rh = rhand[i].copy()
        
        # Fix lệch hông (Cộng dồn tọa độ bàn tay vào cổ tay)
        lw_pos = p[15]
        rw_pos = p[16]
        if not np.isnan(lw_pos).any() and not (lw_pos == 0).all():
            lh = lh + lw_pos
        if not np.isnan(rw_pos).any() and not (rw_pos == 0).all():
            rh = rh + rw_pos
            
        combined = np.vstack([p, lh, rh]) # (75, 3)
        clip_frames.append(combined)
        
    all_processed_clips.append(np.array(clip_frames))

if not all_processed_clips:
    raise ValueError("Không có dữ liệu hợp lệ nào được tải từ danh sách .npz!")

# Tiến hành ghép nối các clip lại với nhau có chèn đoạn chuyển tiếp (Transition)
ntr = max(0, int(round(TRANSITION_SECONDS * global_fps)))
final_timeline_frames = []
segment_info = [] # Lưu mốc thời gian để hiển thị tên từ trên màn hình

cursor = 0
for idx, clip in enumerate(all_processed_clips):
    seg_start = cursor
    
    if idx > 0 and ntr > 0:
        # Lấy khung hình cuối của clip trước và khung hình đầu của clip hiện tại để nội suy chuyển tiếp
        prev_clip = all_processed_clips[idx - 1]
        start_frame = prev_clip[-1]
        end_frame = clip[0]
        
        for t_step in range(1, ntr + 1):
            w = ease(t_step / (ntr + 1))
            interpolated_frame = start_frame * (1 - w) + end_frame * w
            final_timeline_frames.append(interpolated_frame)
            cursor += 1
            
    # Thêm toàn bộ khung hình của clip hiện tại vào timeline chung
    for frame in clip:
        final_timeline_frames.append(frame)
        cursor += 1
        
    segment_info.append({
        "start": seg_start,
        "end": cursor,
        "name": Path(NPZ_FILE_LIST[idx]).stem
    })

full_frames = np.array(final_timeline_frames)
total_frames = len(full_frames)

# ==========================================
# 5. TÍNH TOÁN TỶ LỆ (SCALE) CHO TOÀN TIMELINE
# ==========================================
valid_mask = ~np.isnan(full_frames[:, :, 0])
valid_x = full_frames[:, :, 0][valid_mask]
valid_y = full_frames[:, :, 1][valid_mask]

x_min, x_max = np.min(valid_x), np.max(valid_x)
y_min, y_max = np.min(valid_y), np.max(valid_y)

scale_x = (WIDTH - 2 * PADDING) / (x_max - x_min + 1e-6)
scale_y = (HEIGHT - 2 * PADDING) / (y_max - y_min + 1e-6)
scale = min(scale_x, scale_y)

offset_x = PADDING - x_min * scale + (WIDTH - 2 * PADDING - (x_max - x_min) * scale) / 2
offset_y = PADDING - y_min * scale + (HEIGHT - 2 * PADDING - (y_max - y_min) * scale) / 2

# ==========================================
# 6. PHÁT HOẠT ẢNH LIÊN TỤC BẰNG OPENCV
# ==========================================
delay = int(1000 / global_fps)
window_name = "Mo phong Chuoi .NPZ Lien tuc"
print(f"Đang phát chuỗi hoạt ảnh ({total_frames} frames tổng cộng, {global_fps} FPS). Bấm 'Esc' hoặc 'q' để thoát.")

frame_idx = 0
while frame_idx < total_frames:
    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    curr_points = full_frames[frame_idx]
    screen_pts = {}

    # Xác định xem khung hình hiện tại thuộc từ (segment) nào để hiển thị tên
    current_word = "Unknown"
    for seg in segment_info:
        if seg["start"] <= frame_idx < seg["end"]:
            current_word = seg["name"]
            break

    # Chiếu tọa độ 3D lên màn hình (bỏ qua phần chân từ 25 -> 32)
    for i in range(len(curr_points)):
        if 25 <= i <= 32:
            continue
        x, y = curr_points[i, 0], curr_points[i, 1]
        
        if math.isnan(x) or math.isnan(y) or (x == 0.0 and y == 0.0):
            continue
            
        px = int(x * scale + offset_x)
        py = int(y * scale + offset_y)
        
        screen_pts[i] = (px, py)
        cv2.circle(img, (px, py), radius=2, color=(255, 255, 255), thickness=-1)

    # Vẽ xương nối
    for u, v, color in connections:
        if u in screen_pts and v in screen_pts:
            cv2.line(img, screen_pts[u], screen_pts[v], color, thickness=2)

    # Hiển thị thông tin tên từ và số khung hình lên giao diện
    cv2.putText(img, f"Gloss: {current_word}", (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
    cv2.putText(img, f"Frame: {frame_idx+1}/{total_frames}", (20, 80), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)

    cv2.imshow(window_name, img)

    # Kiểm tra đóng cửa sổ hoặc phím thoát
    if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
        break

    key = cv2.waitKey(delay) & 0xFF
    if key == 27 or key == ord('q'):
        break

    frame_idx += 1
    
    # Nếu muốn video tự động chạy lại từ đầu khi kết thúc chuỗi:
    # if frame_idx >= total_frames: frame_idx = 0

cv2.destroyAllWindows()

[CẢNH BÁO] Không tìm thấy file: C:/Users/ADMIN/qh/processed/processed/npz_output/Yêu thương_007340.npz, bỏ qua.
Đang phát chuỗi hoạt ảnh (196 frames tổng cộng, 30.0 FPS). Bấm 'Esc' hoặc 'q' để thoát.


In [1]:
import json
from pathlib import Path

MAPPING_JSON = "./New folder/anhxa.json"   # đường dẫn file ánh xạ
NPZ_DIR = Path("./New folder/npz")    # thư mục chứa các file .npz

with open(MAPPING_JSON, encoding="utf-8") as f:
    mapping = json.load(f)

# Tập tên file npz hiện có (bỏ đuôi)
npz_stems = {p.stem for p in NPZ_DIR.glob("*.npz")}

missing = []   # có trong mapping nhưng không có npz
for item in mapping:
    stem = Path(item["VIDEO"]).stem          # W00636.mp4 -> W00636
    if stem not in npz_stems:
        missing.append(item)

# Các npz không có trong mapping (kiểm tra chiều ngược lại)
mapped_stems = {Path(i["VIDEO"]).stem for i in mapping}
extra = sorted(npz_stems - mapped_stems)

# Nhóm theo label: label nào KHÔNG có file npz nào
from collections import defaultdict
label_to_stems = defaultdict(list)
for item in mapping:
    label_to_stems[item["LABEL"]].append(Path(item["VIDEO"]).stem)

labels_without_any_npz = [
    lb for lb, stems in label_to_stems.items()
    if not any(s in npz_stems for s in stems)
]

print(f"Tổng video trong mapping: {len(mapping)}")
print(f"Tổng file npz: {len(npz_stems)}")
print(f"\nVideo thiếu npz: {len(missing)}")
for m in missing:
    print(f"  {m['VIDEO']} -> {m['LABEL']}")
print(f"\nLabel hoàn toàn không có npz nào: {len(labels_without_any_npz)}")
for lb in labels_without_any_npz:
    print(f"  {lb}")
print(f"\nFile npz thừa (không có trong mapping): {len(extra)}")
for e in extra:
    print(f"  {e}.npz")

Tổng video trong mapping: 4645
Tổng file npz: 4362

Video thiếu npz: 282
   -> An ủi
   -> Anh
   -> Ba lô
   -> Bia
   -> Biết
   -> Biếu tặng
   -> Bà ngoại
   -> Bà nội
   -> Bàn phím
   -> Bác
   -> Bác sĩ
   -> Bánh bao
   -> Bánh chưng
   -> Bánh tét
   -> Bánh xèo
   -> Bây giờ
   -> Bóng bàn
   -> Bóng chuyền
   -> Bóng rổ
   -> Bóng đá
   -> Bún
   -> Bút chì
   -> Băn khoăn
   -> Bạn thân
   -> Bảng
   -> Bảo vệ
   -> Bận
   -> Bế mạc
   -> Bệnh viện
   -> Bộ y tế
   -> Ca sĩ
   -> Cao (đồ vật)
   -> Cay
   -> Chiều
   -> Cho
   -> Chua
   -> Cháu
   -> Chân
   -> Chú
   -> Chúng ta
   -> Chơi cờ
   -> Chấp nhận
   -> Chậm lại
   -> Chật
   -> Chết
   -> Chồng
   -> Chợ
   -> Chủ nhật
   -> Con trâu
   -> Cà phê
   -> Cái bàn
   -> Cái cửa
   -> Cái ghế
   -> Cái quần
   -> Cái đèn
   -> Cám dỗ
   -> Có
   -> Có thể
   -> Cô
   -> Công ty
   -> Cũ
   -> Cười
   -> Cảm thấy
   -> Cảm ơn
   -> Cần
   -> Cầu lông
   -> Cậu
   -> Cứng
   -> Diễn viên
   -> Dây chuyền
   -> Dép
  